# Player speed over the tactical timeline

Two graphs for inspecting player motion as a possible dead-ball feature:
1. Mean, median, highest and lowest player speed across both teams.
2. Total player speed (sum of scalar speeds), alongside valid-speed track coverage.

Uses the repository's NumPy, pandas and Matplotlib dependencies. No ground-truth labels are loaded or used to choose thresholds. Run all cells from the repo root or `starter/` using the repo's Python environment.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "games.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "games.py").is_file():
    raise RuntimeError("Run this notebook from the repo root or starter/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from games import GAME, GAMES, asset_paths
from starter.player_speed import load_speed_table, plot_speed_timelines, speed_metadata

metadata = speed_metadata(asset_paths()["atd_meta"])
print("Speed units declared by source XML:", metadata["unit"])
print("Ball height channel supplied:", metadata["ballHeightAvailable"])


## Definitions and missing data

Statistics use visible track slots with finite x/y and finite, nonnegative speed. Missing speeds are excluded, not replaced with zero. With no valid speeds, mean, median, highest, lowest and total are all missing. A valid measured zero remains zero. Track slots are not persistent identities and duplicates may exist.

The delivered XML declares `speed_kph` / `kph`, while DATA.md describes m/s. These graphs preserve raw values and label them **km/h according to the XML**; no conversion or physical calibration is inferred. The viewer uses the identical aggregation helper. Total speed depends on the number of observed slots, so the second graph also shows coverage.

Kloppy creates a `ball_z` column through its generic Point3D representation, but this export declares only x, y and s: z is None even where the ball is tracked. Its third raw ball value is speed, not height.


In [ ]:
speeds = load_speed_table()
assert len(speeds) == GAMES[GAME]["n_video_frames"]
assert np.array_equal(speeds["frame"].to_numpy(), np.arange(len(speeds)))
print(f"Loaded {len(speeds):,} video frames")
print(f"Frames without any valid player speed: {(speeds['count'] == 0).sum():,}")
display(speeds[["frame", "seconds", "mean", "median", "min", "max", "total", "count", "tracked"]].head())


## Plot controls

`BIN_SECONDS = 1` averages the per-frame summaries within each second for readability. The median curve is the time-average of the per-frame medians, not a pooled median. Highest and lowest curves likewise average each frame's maximum and minimum; they do not show the absolute extremes of the whole bin. The shaded band spans these two curves. Set `BIN_SECONDS = 1 / 25` to show individual frames, preferably with a smaller time window. A bin with no valid observations remains a gap.

Times are minutes from the start of the tactical video, not the match clock. Gray shading marks frames outside the graded halves. Set `START_MINUTE` and `END_MINUTE` to inspect a period, for example 7 to 12. These display choices are not a trained decision threshold.


In [ ]:
BIN_SECONDS = 1
START_MINUTE = 0
END_MINUTE = None  # None = end of the recording

plt.ioff()  # Display exactly the two figures below, with no duplicate auto-display.
fig_mean_median, fig_total = plot_speed_timelines(
    speeds, bin_seconds=BIN_SECONDS,
    start_minute=START_MINUTE, end_minute=END_MINUTE,
)


In [ ]:
display(fig_mean_median)
plt.close(fig_mean_median)


In [ ]:
display(fig_total)
plt.close(fig_total)


## Interpretation

Low motion may help identify stoppages, but these graphs alone do not establish reliability: live play can be slow, and players can run while the ball is dead. Low totals can also result from occlusion or missing tracks. Inspect the valid-speed count alongside any dip and compare with the synchronized video and automatic tags.

Evaluate any eventual classifier on appropriately labelled **other matches**. Under RULES.md, do not create hand labels for this match or use the five public minutes to train or tune a threshold. No automatic dead-ball claim is made by this notebook.

Generated figures are saved below into the existing gitignored `evaluation/reports/` location. Notebook outputs should be cleared before sharing the repository.


In [ ]:
output_dir = ROOT / "evaluation" / "reports" / "player_speed"
output_dir.mkdir(parents=True, exist_ok=True)
fig_mean_median.savefig(output_dir / "mean_median_speed.png", dpi=170, facecolor="white")
fig_total.savefig(output_dir / "total_speed.png", dpi=170, facecolor="white")
print("Saved graphs to", output_dir)
